In [6]:
import pandas as pd
import ollama
from langchain_core.tools import tool
from pydantic import BaseModel

In [141]:

def get_all_inventory():
    """function returns the all products inventory."""
    inventory=[
        {'name':'hp laptop',
         'price':20000,
         'available_stock':4},
         {'name':'dell laptop',
          'price':25000,
          'available_stock':33},
         {'name':'lenovo laptop',
          'price':50000,
          'available_stock':21},
         {'name':'samsung s26 ultrs',
          'price':150000,
          'available_stock':87},
         {'name':'pixel 8 pro',
          'price':68000,
          'available_stock':50}
    ]

    return inventory

In [142]:
def get_product_inventory(product_name: str):
    """the function return the available stock of the product by taking product name as input and return json object of {'name','price','available_stock'} and if passed product not found then it is return string that product not found"""
    # inventory = [
    #     {'name': 'hp laptop', 'price': 2000, 'available_stock': 4},
    #     {'name': 'dell laptop', 'price': 2500, 'available_stock': 5},
    #     {'name': 'lenovo laptop', 'price': 2200, 'available_stock': 3},
    #     {'name': 'hp mouse', 'price': 500, 'available_stock': 10},
    #     {'name': 'hp keyboard', 'price': 800, 'available_stock': 6}
    # ]

    inventory=get_all_inventory()

    # Search for the product by name
    for product in inventory:
        if product['name'].lower() == product_name.lower():
            return product

    return "Product not found"


In [122]:
messages=[
    {'role':'system','content':'you are a helpful ecommerce assistent'},
    # {'role':'user','content':'who is the prime minister of india. answer in one line'}
    # {'role':'user','content':'give the inventory  of all products'}
    {'role':'user','content':'how many lenovo laptop available'}


]

In [123]:
tools=[get_all_inventory,get_product_inventory]

In [124]:
llm=ollama.chat(model='gemma4:31b-cloud',messages=messages,tools=tools,think='low')

In [125]:
llm

ChatResponse(model='gemma4:31b', created_at='2026-10-17T16:09:43.418390513Z', done=True, done_reason='stop', total_duration=303078208, load_duration=None, prompt_eval_count=148, prompt_eval_duration=None, eval_count=21, eval_duration=None, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_product_inventory', arguments={'product_name': 'lenovo laptop'}))]), logprobs=None)

In [126]:
llm['message']['tool_calls']

[ToolCall(function=Function(name='get_product_inventory', arguments={'product_name': 'lenovo laptop'}))]

In [135]:
messages=[
    {'role':'system','content':'you are a helpful ecommerce assistent'},

    {'role':'user','content':'how many lenovo laptop stock available'}


]

tools=[get_all_inventory,get_product_inventory]
llm=ollama.chat(model='gemma4:31b-cloud',messages=messages,tools=tools)
for tool_call in llm['message']['tool_calls']:

    if tool_call['function']['name']=='get_all_inventory':
        response=get_all_inventory()
        print(response)
        messages.append({'role':'tool','tool_name':tool_call['function']['name'],'content':str(response)})
    if tool_call['function']['name']=='get_product_inventory':
            response=get_product_inventory(**tool_call['function']['arguments'])
            print(response)
            messages.append({'role':'tool','tool_name':tool_call['function']['name'],'content':str(response)})

llm=ollama.chat(model='gemma4:31b-cloud',messages=messages,think=None)
        

{'name': 'lenovo laptop', 'price': 2200, 'available_stock': 3}


In [ ]:
# llm=ollama.chat(model='gemma4:31b-cloud',messages=messages,think='low')

In [136]:
llm.message

Message(role='assistant', content="I do not have direct access to your company's real-time inventory database or warehouse management system.\n\nTo give you the exact number of Lenovo laptops in stock, **please provide one of the following:**\n\n1.  **Connect me to your inventory API/database** (if you are a developer setting up this assistant).\n2.  **Upload a CSV or Excel file** containing your current stock list.\n3.  **Tell me which specific model** you are looking for (e.g., *ThinkPad X1 Carbon* or *IdeaPad 3*), and I can help you search for it if the data has been shared with me.\n\n**If you are a customer asking about a specific store:**\nPlease let me know which website or retail store you are shopping at, and I can guide you on how to check the availability on their product page!", thinking=None, images=None, tool_name=None, tool_calls=None)

In [137]:
messages

[{'role': 'system', 'content': 'you are a helpful ecommerce assistent'},
 {'role': 'user', 'content': 'how many lenovo laptop stock available'},
 {'role': 'tool',
  'tool_name': 'get_product_inventory',
  'content': "{'name': 'lenovo laptop', 'price': 2200, 'available_stock': 3}"}]

In [143]:

import ollama
import json

messages = [
    {
        "role": "system",
        "content": "You are a helpful ecommerce assistant. "
                   "Use inventory tools to answer product availability "
                   "questions. Summarize the inventory results clearly."
    },
    # {
    #     "role": "user",
    #     "content": "give me hp laptop available stock"
    # }
    {
            "role": "user",
            "content": "give me entire products inventory"
        }
]

tools = [get_all_inventory, get_product_inventory]

# STEP 1: First LLM call
llm = ollama.chat(
    model="gemma4:31b-cloud",
    messages=messages,
    tools=tools,
    think="low"
)

# STEP 2: Append assistant's tool-call message
messages.append(llm.message)

# STEP 3: Execute requested tools
if llm.message.tool_calls:

    for tool_call in llm.message.tool_calls:

        tool_name = tool_call.function.name
        tool_args = tool_call.function.arguments

        if tool_name == "get_all_inventory":
            response = get_all_inventory()
            print("Tool Result:", response)

            messages.append({
                "role": "tool",
                "tool_name": tool_name,
                "content": json.dumps(response, default=str)
            })

        elif tool_name == "get_product_inventory":
            response = get_product_inventory(**tool_args)
            print("Tool Result:", response)

            messages.append({
                "role": "tool",
                "tool_name": tool_name,
                "content": json.dumps(response, default=str)
            })

    # STEP 4: Generate final answer from tool results
    final_response = ollama.chat(
        model="gemma4:31b-cloud",
        messages=messages,
        think="low"
    )

    print("\nFinal Answer:")
    print(final_response.message.content)

else:
    print(llm.message.content)


Tool Result: [{'name': 'hp laptop', 'price': 20000, 'available_stock': 4}, {'name': 'dell laptop', 'price': 25000, 'available_stock': 33}, {'name': 'lenovo laptop', 'price': 50000, 'available_stock': 21}, {'name': 'samsung s26 ultrs', 'price': 150000, 'available_stock': 87}, {'name': 'pixel 8 pro', 'price': 68000, 'available_stock': 50}]

Final Answer:
Here is the current product inventory:

| Product Name | Price | Available Stock |
| :--- | :--- | :--- |
| **Samsung S26 Ultra** | 150,000 | 87 |
| **Pixel 8 Pro** | 68,000 | 50 |
| **Lenovo Laptop** | 50,000 | 21 |
| **Dell Laptop** | 25,000 | 33 |
| **HP Laptop** | 20,000 | 4 |
